# ARGUS · Session 3 — 1280px Boost + Polish + Eval + Export

**Before starting:** Upload `argus_s2.pt` from Session 2 to `/kaggle/working/`

**Training order (publication-aligned):**
1. **1280px boost** (10 epochs, batch=8, lr0=2e-5) — resolution adaptation BEFORE final polish
2. **640px polish** (20 epochs, `close_mosaic=8`) — last 8 epochs on clean images
3. **Held-out evaluation** — locked 2k UA-DETRAC images, never seen during training
4. **ONNX export** — Jetson/CPU inference pipeline

**Primary metric: mAP50-95** (COCO standard, captures localisation precision for TTC)
**Time budget: ~20 hr on Kaggle T4×2** (boost ~9 hr + polish ~7 hr + setup ~4 hr)

---

## Attach same Kaggle datasets as Sessions 1 & 2

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
from pathlib import Path
import torch, os, json as _json

NC      = 5
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle']

WORK     = Path('/kaggle/working')
OUT_DIR  = WORK / 'argus_data'
MERGED   = OUT_DIR / 'merged'
HARD_DIR = OUT_DIR / 'hard'
HELD_DIR = OUT_DIR / 'held'
RUNS_DIR = WORK / 'runs'
INPUT    = Path('/kaggle/input')
yaml_path  = MERGED / 'data.yaml'
held_yaml  = HELD_DIR / 'data.yaml'

for d in [MERGED/'train/images', MERGED/'train/labels',
          MERGED/'valid/images', MERGED/'valid/labels',
          HARD_DIR/'train/images', HARD_DIR/'train/labels',
          HARD_DIR/'valid/images', HARD_DIR/'valid/labels',
          HELD_DIR/'valid/images', HELD_DIR/'valid/labels',
          RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

UPLOADED_S2  = WORK / 'argus_s2.pt'
S3_1280_BEST = RUNS_DIR/'s3_1280'/'argus_s3_1280'/'weights'/'best.pt'
S3_1280_LAST = RUNS_DIR/'s3_1280'/'argus_s3_1280'/'weights'/'last.pt'
S3P_BEST     = RUNS_DIR/'s3p'/'argus_s3p'/'weights'/'best.pt'
S3P_LAST     = RUNS_DIR/'s3p'/'argus_s3p'/'weights'/'last.pt'

DEVICE = ','.join(str(i) for i in range(torch.cuda.device_count())) or 'cpu'
BATCH  = 16
IMGSZ  = 640

KGL_USER = os.environ.get('KAGGLE_USERNAME','')
KGL_KEY  = os.environ.get('KAGGLE_KEY','')
if KGL_USER and KGL_KEY:
    kdir = Path.home()/'.kaggle'; kdir.mkdir(exist_ok=True)
    (kdir/'kaggle.json').write_text(_json.dumps({'username':KGL_USER,'key':KGL_KEY}))
    (kdir/'kaggle.json').chmod(0o600)
    print(f'Kaggle creds: {KGL_USER}')
else:
    print('Kaggle creds NOT set — BDD100K + UA-DETRAC will be skipped')

print(f'DEVICE={DEVICE}  BATCH={BATCH}  IMGSZ={IMGSZ}')


In [ ]:
# ── Kaggle API credentials — paste here if Secrets don't work ────────────────
# Get your key from https://kaggle.com/settings -> API -> Create New Token
import os, json as _j
from pathlib import Path

KAGGLE_USER = ''   # <- paste your Kaggle username
KAGGLE_KEY  = ''   # <- paste your API key

# Fall back to environment Secrets if not hardcoded
if not KAGGLE_USER: KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', '')
if not KAGGLE_KEY:  KAGGLE_KEY  = os.environ.get('KAGGLE_KEY', '')

if KAGGLE_USER and KAGGLE_KEY:
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'kaggle.json').write_text(_j.dumps({'username': KAGGLE_USER, 'key': KAGGLE_KEY}))
    (kdir / 'kaggle.json').chmod(0o600)
    # Propagate into variables used by dataset cells
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    print(f'Kaggle: authenticated as {KAGGLE_USER}')
else:
    print('Kaggle: no credentials -- API downloads disabled, using attached datasets only')


In [ ]:
# ── Install + GPU check ──────────────────────────────────────────────────────
import subprocess, sys, torch

n_gpu = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics>=8.4.0', 'pyyaml', 'pycocotools', 'kaggle', 'pillow'], check=True)
import ultralytics; ultralytics.checks()

In [ ]:
# ── Pre-flight: what's in /kaggle/input/ ──────────────────────────────────────
import os
from pathlib import Path

print('Mounted datasets in /kaggle/input/datasets/:')
ds_root = INPUT / 'datasets'
mounted_names = set()
if ds_root.exists():
    for user_dir in sorted(ds_root.iterdir()):
        if not user_dir.is_dir(): continue
        print(f'  {user_dir.name}/')
        mounted_names.add(user_dir.name.lower())
        for ds_dir in sorted(user_dir.iterdir()):
            if not ds_dir.is_dir(): continue
            try:
                mb = sum(f.stat().st_size for f in ds_dir.rglob('*') if f.is_file()) // 1024**2
                print(f'    {ds_dir.name}/  (~{mb} MB)')
            except Exception:
                print(f'    {ds_dir.name}/')
elif INPUT.exists():
    for d in sorted(INPUT.iterdir()):
        if d.is_dir():
            mounted_names.add(d.name.lower())
            print(f'  {d.name}/')

print()
s2_ok  = UPLOADED_S2.exists() or next(INPUT.rglob('argus_s2.pt'), None) is not None
idd_ok = any('idd' in n for n in mounted_names) or next(INPUT.rglob('idd20k_final'), None) is not None
bdd_ok = any('bdd' in n for n in mounted_names) or bool(KGL_USER)
print(f'  argus_s2.pt : {"OK" if s2_ok  else "MISSING -- upload to /kaggle/working/"}')
print(f'  IDD dataset : {"OK" if idd_ok  else "MISSING -- attach abhishekprajapat/idd-20k"}')
print(f'  BDD100K     : {"OK" if bdd_ok else "MISSING -- attach or set Kaggle creds"}')
print()
if not s2_ok:
    print('WARNING: argus_s2.pt is required -- download from Session 2 Output tab')
elif not idd_ok:
    print('WARNING: IDD dataset missing')
else:
    print('Required datasets look OK -- proceed')


In [ ]:
# ── UA-DETRAC → YOLO (overhead CCTV) ─────────────────────────────────────────
# Adds overhead traffic camera viewpoint — critical for near-miss scenarios.
# car→0  bus→2  van/truck→3  motorbike→1
# 2k held-out images locked per session (random.seed(42) — deterministic).
import subprocess, os, re, random, xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

random.seed(42)
FLAG = OUT_DIR / '.uadetrac_done'
if FLAG.exists():
    print('UA-DETRAC: already done — skipping')
else:
    DETRAC_MAP = {'car':0,'van':3,'truck':3,'bus':2,'motorbike':1,'motorcycle':1,'others':0}
    IMG_W, IMG_H = 960, 540  # UA-DETRAC standard resolution

    DETRAC_ROOT = next((INPUT/s for s in ['ua-detrac','uadetrac','ua_detrac','detrac-training']
                        if (INPUT/s).exists()), None)
    # Also search nested datasets/<user>/<ds>/ (Kaggle merged dataset mounting)
    if DETRAC_ROOT is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir(): continue
                if DETRAC_ROOT: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    xml_count = sum(1 for _ in ds_dir.rglob('*.xml'))
                    if xml_count > 50:
                        DETRAC_ROOT = ds_dir
                        print(f'UA-DETRAC: found at datasets/{user_dir.name}/{ds_dir.name}/ ({xml_count} XMLs)')
                        break
    if DETRAC_ROOT is None and KGL_USER:
        RAW = WORK/'_detrac_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW/'.downloaded'
        SLUGS = ['datatang/ua-detrac','dtrackers/ua-detrac','fanbyprince/ua-detrac',
                 'robustchicken/ua-detrac','adamdodge/ua-detrac']
        if dl_flag.exists():
            print('UA-DETRAC: already downloaded'); DETRAC_ROOT = RAW
        else:
            for slug in SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); DETRAC_ROOT = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:100]}')
            else:
                print('  All UA-DETRAC slugs failed — skipping')

    if DETRAC_ROOT is None:
        print('UA-DETRAC: not found — set Kaggle creds or attach dataset')
    else:
        seq_xmls = sorted(DETRAC_ROOT.rglob('*.xml'))
        xml_by_seq = {x.stem: x for x in seq_xmls}
        print(f'Found {len(seq_xmls)} sequence XMLs')

        seq_ann = {}
        for xp in tqdm(seq_xmls, desc='Parse XMLs'):
            try: root_el = ET.parse(xp).getroot()
            except ET.ParseError: continue
            frames = {}
            for frame in root_el.findall('.//frame'):
                fn = int(frame.get('num', 0))
                boxes = []
                for tgt in frame.findall('.//target'):
                    attr = tgt.find('attribute')
                    vtype = (attr.get('vehicle_type','') if attr is not None else '').lower()
                    ac = DETRAC_MAP.get(vtype)
                    if ac is None: continue
                    box = tgt.find('box')
                    if box is None: continue
                    try:
                        l = float(box.get('left',0)); t = float(box.get('top',0))
                        bw = float(box.get('width',0)); bh = float(box.get('height',0))
                    except ValueError: continue
                    if bw <= 0 or bh <= 0: continue
                    cx = max(.001, min(.999, (l+bw/2)/IMG_W))
                    cy = max(.001, min(.999, (t+bh/2)/IMG_H))
                    nw = max(.001, min(.999, bw/IMG_W))
                    nh = max(.001, min(.999, bh/IMG_H))
                    boxes.append((ac, cx, cy, nw, nh))
                if boxes: frames[fn] = boxes
            if frames: seq_ann[xp.stem] = frames

        all_imgs = sorted(DETRAC_ROOT.rglob('img*.jpg')) + sorted(DETRAC_ROOT.rglob('img*.png'))
        imgs_shuffled = list(all_imgs); random.shuffle(imgs_shuffled)
        held_set = set(str(p) for p in imgs_shuffled[:2000])

        HELD_DIR.mkdir(parents=True, exist_ok=True)
        (HELD_DIR/'valid'/'images').mkdir(parents=True, exist_ok=True)
        (HELD_DIR/'valid'/'labels').mkdir(parents=True, exist_ok=True)

        nv = n_held = 0
        for img in tqdm(all_imgs, desc='UA-DETRAC'):
            seq_name = img.parent.name
            m = re.search(r'(\d+)', img.stem[-8:])
            fn = int(m.group(1)) if m else 0
            ann = seq_ann.get(seq_name, {}).get(fn)
            if not ann: continue
            lines = [f'{ac} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}' for ac,cx,cy,nw,nh in ann]
            stem = f'detrac_{seq_name}_{img.stem}'
            if str(img) in held_set:
                hi = HELD_DIR/'valid'/'images'; hl = HELD_DIR/'valid'/'labels'
                link = hi/(stem+img.suffix)
                if not link.exists(): _imglink(img, link)
                (hl/(stem+'.txt')).write_text('\n'.join(lines))
                n_held += 1
            else:
                dst = 'train' if random.random() < 0.9 else 'valid'
                di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
                link = di/(stem+img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl/(stem+'.txt')).write_text('\n'.join(lines))
                nv += 1

        (HELD_DIR/'data.yaml').write_text(
            f'path: {HELD_DIR.resolve()}\nval: valid/images\nnc: {NC}\nnames: {CLASSES}\n')
        print(f'UA-DETRAC: {nv:,} train/val | {n_held:,} held-out locked')
        FLAG.touch(); print('UA-DETRAC: done')


In [ ]:
# ── BDD100K → YOLO ───────────────────────────────────────────────────────────
# motorcycle ×3 oversample, bicycle ×2 oversample for class balance.
# Checks /kaggle/input/datasets/ first (attached), falls back to Kaggle API.
import os, subprocess, yaml
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

FLAG = OUT_DIR / '.bdd_done'
if FLAG.exists():
    print('BDD100K: already done — skipping')
else:
    name_to_argus = {'car':0,'automobile':0,'motorcycle':1,'motorbike':1,'motor':1,
                     'bus':2,'truck':3,'van':3,'bicycle':4,'bike':4}

    BDD_DIR = None

    # 1. Top-level slug check
    ATTACHED_SLUGS = ['bdd100k-yolo-format','bdd100k-dataset','bdd100k','bdd100k-yolo',
                      'bdd100k-yolo-v8','bdd-100k','bdd100k_yolo','bdd_100k','bdd100k_yolo_format']
    for slug in ATTACHED_SLUGS:
        candidate = INPUT / slug
        if candidate.exists() and any(candidate.rglob('*.jpg')):
            BDD_DIR = candidate
            print(f'BDD100K: found attached at {BDD_DIR}')
            break

    # 2. Nested datasets/<user>/<ds>/ search
    if BDD_DIR is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir(): continue
                if BDD_DIR: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    # Check data.yaml for BDD-specific class names
                    for yf in ds_dir.rglob('data.yaml'):
                        try:
                            cfg = yaml.safe_load(yf.read_text())
                            names_list = cfg.get('names', [])
                            if isinstance(names_list, dict): names_list = list(names_list.values())
                            bdd_markers = {'rider', 'traffic light', 'traffic sign', 'motor'}
                            if bdd_markers & {n.strip().lower() for n in names_list}:
                                BDD_DIR = ds_dir
                                print(f'BDD100K: found at datasets/{user_dir.name}/{ds_dir.name}/ (BDD class names)')
                                break
                        except Exception: pass
                    if BDD_DIR: break
                    # Fallback: large dataset by jpg count
                    if not BDD_DIR and 'idd' not in str(ds_dir).lower() and 'argus' not in str(ds_dir).lower():
                        jpg_n = sum(1 for _ in ds_dir.rglob('*.jpg'))
                        if jpg_n > 50000:
                            BDD_DIR = ds_dir
                            print(f'BDD100K: found at datasets/{user_dir.name}/{ds_dir.name}/ ({jpg_n} images, assumed BDD)')
                            break

    # 3. API download fallback
    if BDD_DIR is None and KGL_USER:
        RAW = WORK / '_bdd_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        API_SLUGS = ['farzadnekouei/bdd100k-yolo-format','awsaf49/bdd100k-dataset',
                     'a2zcode/bdd100k','gautamchettiar/bdd100k','a7madmostafa/bdd100k-yolo']
        if dl_flag.exists():
            print('BDD100K: already downloaded'); BDD_DIR = RAW
        else:
            for slug in API_SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); BDD_DIR = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:120]}')
            if BDD_DIR is None:
                print('  All BDD100K slugs failed — skipping')

    if BDD_DIR is None:
        print('BDD100K: not found — attach dataset or set Kaggle Secrets')
    else:
        bdd_map = None
        for yf in sorted(BDD_DIR.rglob('*.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                m = {i: name_to_argus[n.lower().strip()] for i, n in enumerate(names)
                     if n.lower().strip() in name_to_argus}
                if m: bdd_map = m; print(f'  BDD map (from {yf.name}): {m}'); break
            except Exception: pass
        if bdd_map is None:
            # BDD100K standard 10-class ordering:
            # pedestrian(0) rider(1) car(2) truck(3) bus(4) train(5)
            # motorcycle(6) bicycle(7) traffic light(8) traffic sign(9)
            bdd_map = {2:0, 6:1, 4:2, 3:3, 7:4}
            print(f'  BDD map: fallback {bdd_map}')

        for split, dst in [('train','train'), ('val','valid')]:
            idirs = [d for d in BDD_DIR.rglob('images') if split in str(d)]
            ldirs = [d for d in BDD_DIR.rglob('labels') if split in str(d)]
            if not idirs: print(f'  BDD {split}: not found — skip'); continue
            idir = idirs[0]
            ldir = ldirs[0] if ldirs else idir.parent.parent/'labels'/split
            di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
            nv = nm = nb_cnt = 0
            for img in tqdm(list(idir.glob('*.*')), desc=f'BDD {split}'):
                lp = ldir / (img.stem + '.txt')
                if not lp.exists(): continue
                lines, has_m, has_b = [], False, False
                for row in lp.read_text().strip().splitlines():
                    p = row.split()
                    if not p: continue
                    try: orig = int(p[0])
                    except ValueError: continue
                    if orig in bdd_map:
                        ac = bdd_map[orig]; lines.append(f'{ac} ' + ' '.join(p[1:]))
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                if not lines: continue
                stem = f'bdd_{img.stem}'
                link = di / (stem + img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl / (stem + '.txt')).write_text('\n'.join(lines))
                nv += 1
                if has_m:
                    nm += 1
                    for k in range(2):
                        ml = di / (f'bdd_{img.stem}_m{k}' + img.suffix)
                        if not ml.exists(): _imglink(img, ml)
                        (dl / (f'bdd_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                if has_b:
                    nb_cnt += 1
                    bl = di / (f'bdd_{img.stem}_b0' + img.suffix)
                    if not bl.exists(): _imglink(img, bl)
                    (dl / (f'bdd_{img.stem}_b0.txt')).write_text('\n'.join(lines))
            print(f'  BDD {split}: {nv} imgs | moto×3: {nm} | bike×2: {nb_cnt}')
        FLAG.touch(); print('BDD100K: done')


In [ ]:
# ── IDD (Indian Driving Dataset) → YOLO ──────────────────────────────────────
# motorcycle/autorickshaw ×4, bicycle ×2 oversample.
import xml.etree.ElementTree as ET
import os, random, yaml, subprocess
from pathlib import Path
from tqdm import tqdm

FLAG = OUT_DIR / '.idd_done'
if FLAG.exists():
    print('IDD: already done -- skipping')
else:
    IDD_NAME_MAP = {
        'car':0, 'auto':0, 'automobile':0,
        'autorickshaw':1, 'auto rickshaw':1, 'three wheeler':1,
        'motorcycle':1, 'motorbike':1, 'two wheeler':1, 'scooter':1,
        'bus':2,
        'truck':3, 'van':3, 'tempo':3, 'vehicle_fallback':3,
        'bicycle':4, 'cycle':4,
    }

    IDD_ROOT = None

    # 1. Known top-level slugs
    for slug in ['idd-20k', 'idd_20k', 'idd-dataset-yolo-format', 'idd', 'idd-detection', 'idd20k', 'idd-yolo']:
        if (INPUT / slug).exists():
            IDD_ROOT = INPUT / slug
            print(f'IDD: found at /kaggle/input/{slug}/')
            break

    # 2. Search for idd20k_final by name (handles datasets/<user>/ merged mounting)
    if IDD_ROOT is None:
        hit = next(INPUT.rglob('idd20k_final'), None)
        if hit and hit.is_dir():
            IDD_ROOT = hit
            print(f'IDD: found idd20k_final at {hit}')

    # 3. Search data.yaml files for IDD-specific class names
    if IDD_ROOT is None:
        for yf in sorted(INPUT.rglob('data.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names_list = cfg.get('names', [])
                if isinstance(names_list, dict): names_list = list(names_list.values())
                idd_markers = {'autorickshaw', 'two wheeler', 'scooter', 'tempo', 'cycle'}
                if idd_markers & {n.strip().lower() for n in names_list}:
                    IDD_ROOT = yf.parent
                    print(f'IDD: found via data.yaml at {yf}')
                    break
            except Exception:
                pass

    # 4. Search for VOC Annotations/ dir
    if IDD_ROOT is None:
        for ann in INPUT.rglob('Annotations'):
            if ann.is_dir() and next(ann.glob('*.xml'), None):
                IDD_ROOT = ann.parent
                print(f'IDD: found VOC dataset at {IDD_ROOT}')
                break

    # 5. API download
    if IDD_ROOT is None and KGL_USER:
        IDD_DL = WORK / '_idd_raw'
        IDD_DL.mkdir(parents=True, exist_ok=True)
        for slug in ['abhishekprajapat/idd-20k', 'ashwinak/idd-dataset-yolo-format']:
            r = subprocess.run(['kaggle', 'datasets', 'download', '-d', slug,
                                '-p', str(IDD_DL), '--unzip'],
                               capture_output=True, text=True)
            if r.returncode == 0:
                IDD_ROOT = IDD_DL
                print(f'IDD: downloaded via {slug}')
                break
            print(f'  {slug}: {r.stderr.strip()[:100]}')

    if IDD_ROOT is None:
        print('IDD: not found')
        mounted = [d.name for d in INPUT.iterdir() if d.is_dir()] if INPUT.exists() else []
        print(f'  Mounted top-level: {mounted}')
        print('  Attach abhishekprajapat/idd-20k as a Kaggle dataset and rerun.')
    else:
        yolo_yamls = list(IDD_ROOT.rglob('data.yaml'))
        yolo_idirs = list(IDD_ROOT.rglob('images'))
        yolo_ldirs = list(IDD_ROOT.rglob('labels'))
        voc_ann    = next(IDD_ROOT.rglob('Annotations'), None)
        voc_img    = next(IDD_ROOT.rglob('JPEGImages'), None)

        direct_splits = []
        for split_kw, dst in [('train', 'train'), ('val', 'valid'), ('valid', 'valid')]:
            sdir = IDD_ROOT / split_kw
            if sdir.exists():
                imgs = list(sdir.glob('*.jpg')) + list(sdir.glob('*.png')) + list(sdir.glob('*.jpeg'))
                lbls = list(sdir.glob('*.txt'))
                if imgs and lbls:
                    direct_splits.append((dst, sdir, sdir))

        if yolo_idirs and yolo_ldirs:
            print('IDD: YOLO format (images/ + labels/ subdirs)')
            idd_map = {}
            if yolo_yamls:
                try:
                    cfg = yaml.safe_load(yolo_yamls[0].read_text())
                    names = cfg.get('names', [])
                    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                    idd_map = {i: IDD_NAME_MAP.get(n.strip().lower()) for i, n in enumerate(names)}
                    idd_map = {k: v for k, v in idd_map.items() if v is not None}
                    print(f'  class map (from {yolo_yamls[0].name}): {idd_map}')
                except Exception as e:
                    print(f'  data.yaml parse failed: {e}')
            if not idd_map:
                sample_lbl = next(IDD_ROOT.rglob('*.txt'), None)
                if sample_lbl:
                    ids = set()
                    for row in sample_lbl.read_text().strip().splitlines()[:20]:
                        p = row.split()
                        if p:
                            try: ids.add(int(p[0]))
                            except ValueError: pass
                    print(f'  No data.yaml -- sample class IDs: {sorted(ids)}')
                idd_map = {i: i for i in range(5)}
                print(f'  class map: identity 0-4')

            for kw, dst in [('train', 'train'), ('val', 'valid'), ('test', 'valid')]:
                idirs = [d for d in yolo_idirs if kw in str(d).lower()]
                ldirs = [d for d in yolo_ldirs if kw in str(d).lower()]
                if not idirs: continue
                idir = idirs[0]
                if ldirs:
                    ldir = ldirs[0]
                elif (IDD_ROOT / 'labels' / kw).exists():
                    ldir = IDD_ROOT / 'labels' / kw
                else:
                    ldir = idir.parent / 'labels'
                print(f'  IDD {kw}: images={idir}  labels={ldir}')
                di = MERGED / dst / 'images'; dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for img in tqdm(list(idir.glob('*.*')), desc=f'IDD {kw}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]
                            lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{kw}_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): os.symlink(img.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{kw}_{img.stem}_m{k}' + img.suffix)
                            if not mlink.exists(): os.symlink(img.resolve(), mlink)
                            (dl / (f'idd_{kw}_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{kw}_{img.stem}_b0' + img.suffix)
                        if not blink.exists(): os.symlink(img.resolve(), blink)
                        (dl / (f'idd_{kw}_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD {kw}: {nv} imgs | moto x4: {nm} | bike x2: {nb}')
                if nv == 0 and list(idir.glob('*.*')):
                    sample = next(idir.glob('*.*'))
                    slp = ldir / (sample.stem + '.txt')
                    print(f'  DEBUG: sample img={sample.name} label_exists={slp.exists()}')
                    if slp.exists():
                        print(f'  DEBUG: label content: {slp.read_text()[:200]}')

        elif direct_splits:
            print('IDD: YOLO format (images directly in train/val dirs)')
            idd_map = {i: i for i in range(5)}
            for dst, idir, ldir in direct_splits:
                di = MERGED / dst / 'images'; dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for img in tqdm(list(idir.glob('*.jpg')) + list(idir.glob('*.png')),
                                desc=f'IDD direct {dst}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]
                            lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{dst}_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): os.symlink(img.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{dst}_{img.stem}_m{k}' + img.suffix)
                            if not mlink.exists(): os.symlink(img.resolve(), mlink)
                            (dl / (f'idd_{dst}_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{dst}_{img.stem}_b0' + img.suffix)
                        if not blink.exists(): os.symlink(img.resolve(), blink)
                        (dl / (f'idd_{dst}_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD direct {dst}: {nv} imgs | moto x4: {nm} | bike x2: {nb}')

        elif voc_ann and voc_img:
            print('IDD: VOC XML format')
            xmls = list(voc_ann.glob('*.xml'))
            random.seed(42); random.shuffle(xmls)
            idx = int(len(xmls) * 0.9)
            for dst, xlist in [('train', xmls[:idx]), ('valid', xmls[idx:])]:
                di = MERGED / dst / 'images'; dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for xp in tqdm(xlist, desc=f'IDD VOC {dst}'):
                    try: root_el = ET.parse(xp).getroot()
                    except ET.ParseError: continue
                    sz = root_el.find('size')
                    if sz is None or sz.find('width') is None: continue
                    W, H = int(sz.find('width').text), int(sz.find('height').text)
                    if W <= 0 or H <= 0: continue
                    lines, has_m, has_b = [], False, False
                    for obj in root_el.findall('object'):
                        nm_raw = obj.find('name').text.strip()
                        ac = IDD_NAME_MAP.get(nm_raw.lower())
                        if ac is None: continue
                        bb = obj.find('bndbox')
                        x1, y1, x2, y2 = (float(bb.find(t).text) for t in ['xmin','ymin','xmax','ymax'])
                        lines.append(
                            f'{ac} {max(.001,min(.999,(x1+x2)/2/W)):.6f}'
                            f' {max(.001,min(.999,(y1+y2)/2/H)):.6f}'
                            f' {max(.001,min(.999,(x2-x1)/W)):.6f}'
                            f' {max(.001,min(.999,(y2-y1)/H)):.6f}')
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                    if not lines: continue
                    ip = next((voc_img/(xp.stem+e) for e in ['.jpg','.jpeg','.png']
                               if (voc_img/(xp.stem+e)).exists()), None)
                    if ip is None: continue
                    stem = f'idd_{xp.stem}'
                    link = di / (stem + ip.suffix)
                    if not link.exists(): os.symlink(ip.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{xp.stem}_m{k}' + ip.suffix)
                            if not mlink.exists(): os.symlink(ip.resolve(), mlink)
                            (dl / (f'idd_{xp.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{xp.stem}_b0' + ip.suffix)
                        if not blink.exists(): os.symlink(ip.resolve(), blink)
                        (dl / (f'idd_{xp.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD VOC {dst}: {nv} | moto x4: {nm} | bike x2: {nb}')
        else:
            print(f'IDD: unrecognised structure at {IDD_ROOT}')
            print(f'  rglob images/: {yolo_idirs[:3]}')
            print(f'  rglob labels/: {yolo_ldirs[:3]}')
        FLAG.touch()
        print('IDD: done')


In [ ]:
# ── KITTI → YOLO ─────────────────────────────────────────────────────────────
# Public wget download (~12 GB). Car→0  Van→3  Truck→3  Cyclist→1  Bus→2
import os, subprocess, zipfile, random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

FLAG = OUT_DIR / '.kitti_done'
if FLAG.exists():
    print('KITTI: already done — skipping')
else:
    RAW  = WORK/'_kitti_raw'; RAW.mkdir(parents=True, exist_ok=True)
    IMGS = RAW/'training'/'image_2'; LBLS = RAW/'training'/'label_2'
    KMAP = {'Car':0,'Van':3,'Truck':3,'Cyclist':1,'Bus':2}

    def _get(url, dst):
        if dst.exists() and dst.stat().st_size > 100_000: return
        subprocess.run(['wget','-q','--show-progress','-O',str(dst),url], check=True)

    kitti_ok = True
    if not IMGS.exists():
        z = RAW/'ki.zip'
        if _get('https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_image_2.zip', z) is not False:
            with zipfile.ZipFile(z) as zf: zf.extractall(RAW)
            z.unlink(missing_ok=True)
        else:
            kitti_ok = False
    if kitti_ok and not LBLS.exists():
        z = RAW/'kl.zip'
        if _get('https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_label_2.zip', z) is not False:
            with zipfile.ZipFile(z) as zf: zf.extractall(RAW)
            z.unlink(missing_ok=True)
        else:
            kitti_ok = False
    if not kitti_ok:
        FLAG.touch(); print('KITTI: skipped (download unavailable)'); raise SystemExit(0)

    all_l = sorted(LBLS.glob('*.txt'))
    random.seed(42); random.shuffle(all_l)
    idx = int(len(all_l)*0.85)
    for dst, llist in [('train',all_l[:idx]),('valid',all_l[idx:])]:
        di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
        nv = nm = 0
        for lp in tqdm(llist, desc=f'KITTI {dst}'):
            ip = IMGS/(lp.stem+'.png')
            if not ip.exists(): continue
            try:
                with Image.open(ip) as im: W, H = im.size
            except Exception: continue
            lines, has_m = [], False
            for row in lp.read_text().strip().splitlines():
                p = row.split()
                if len(p) < 15: continue
                ac = KMAP.get(p[0])
                if ac is None: continue
                x1,y1,x2,y2 = map(float, p[4:8])
                lines.append(f'{ac} {max(.001,min(.999,(x1+x2)/2/W)):.6f}'
                             f' {max(.001,min(.999,(y1+y2)/2/H)):.6f}'
                             f' {max(.001,min(.999,(x2-x1)/W)):.6f}'
                             f' {max(.001,min(.999,(y2-y1)/H)):.6f}')
                if ac==1: has_m=True
            if not lines: continue
            stem = f'kitti_{lp.stem}'
            link = di/(stem+'.png')
            if not link.exists(): os.symlink(ip.resolve(), link)
            (dl/(stem+'.txt')).write_text('\n'.join(lines))
            nv += 1
            if has_m: nm += 1
        print(f'  KITTI {dst}: {nv} imgs | {nm} moto')
    FLAG.touch(); print('KITTI: done')

In [ ]:
# ── Dataset stats + data.yaml ────────────────────────────────────────────────
from pathlib import Path
from collections import Counter

def _count(split):
    ldir = MERGED/split/'labels'
    imgs = len(list((MERGED/split/'images').glob('*.*')))
    stats = Counter()
    for lp in ldir.glob('*.txt'):
        for row in lp.read_text().splitlines():
            p = row.strip().split()
            if p:
                try: stats[int(p[0])] += 1
                except ValueError: pass
    return imgs, stats

print('='*60)
for split in ['train','valid']:
    n, stats = _count(split)
    total = sum(stats.values())
    bad = {k:v for k,v in stats.items() if k<0 or k>=NC}
    print(f'\n{split}: {n:,} images | {total:,} boxes')
    for cid, name in enumerate(CLASSES):
        bar = '█' * min(40, int(40*stats.get(cid,0)/max(total,1)))
        print(f'  {cid} {name:12s}: {stats.get(cid,0):8,}  {bar}')
    if bad:
        raise ValueError(f'OUT-OF-RANGE class IDs in {split}: {bad}')
    print(f'  ✓ all IDs in [0,{NC-1}]')
print('='*60)
n_train = len(list((MERGED/'train'/'images').glob('*.*')))
n_moto  = sum(1 for lp in (MERGED/'train'/'labels').glob('*.txt')
              if any(r.split()[0]=='1' for r in lp.read_text().splitlines() if r.strip()))
assert n_train > 5_000, f'Only {n_train} training images'
assert n_moto  > 500,   f'Only {n_moto} motorcycle images — BDD class map likely broken'
yaml_path.write_text(
    f'path: {MERGED.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n')
print(f'\ndata.yaml → {yaml_path}')
print(yaml_path.read_text())

In [ ]:
# ── 1280px resolution boost (10 epochs) ──────────────────────────────────────
# Runs BEFORE polish — teaches the network fine-grained high-res features.
# Dashcam vehicles at 50-200 m are 20-60 px at 640px; 80-240 px at 1280px.
# Council A: batch=8 (4/GPU), lr0=2e-5 — higher than default for meaningful
#            weight update at resolution shift; dfl=2.0 carried forward.
# ~55 min/epoch on T4×2 → ~9 hr total for 10 epochs.
from ultralytics import YOLO
import shutil, time

# Accept argus_s2b.pt (from S2.5) or argus_s2.pt
for _name in ['argus_s2b.pt', 'argus_s2.pt']:
    if not UPLOADED_S2.exists() and (WORK/_name).exists():
        import shutil as _sh; _sh.copy2(WORK/_name, UPLOADED_S2)
        print(f'Using {_name} as S2 checkpoint')
if not UPLOADED_S2.exists():
    if (RUNS_DIR/'s2b'/'argus_s2b'/'weights'/'best.pt').exists():
        shutil.copy2(RUNS_DIR/'s2b'/'argus_s2b'/'weights'/'best.pt', UPLOADED_S2)
    elif (RUNS_DIR/'s2a'/'argus_s2a'/'weights'/'best.pt').exists():
        shutil.copy2(RUNS_DIR/'s2a'/'argus_s2a'/'weights'/'best.pt', UPLOADED_S2)
    else:
        raise FileNotFoundError(
            'argus_s2.pt not found at /kaggle/working/\n'
            'Download from Session 2 output and upload before running.')
assert yaml_path.exists(), 'Run dataset cells first'

BATCH_1280 = 8 if ',' in DEVICE else 4
print(f'1280px boost: batch={BATCH_1280}  device={DEVICE}')

t0 = time.time()
if S3_1280_LAST.exists():
    print(f'1280px: resuming from {S3_1280_LAST}')
    model = YOLO(str(S3_1280_LAST)); model.train(resume=True)
else:
    print(f'1280px: starting from {UPLOADED_S2}')
    model = YOLO(str(UPLOADED_S2))
    model.train(
        data=str(yaml_path), imgsz=1280, epochs=10, batch=BATCH_1280, device=DEVICE,
        project=str(RUNS_DIR/'s3_1280'), name='argus_s3_1280',
        freeze=0, optimizer='AdamW', lr0=2e-5, lrf=0.1, warmup_epochs=0,
        weight_decay=5e-4, box=9.0, cls=0.3, dfl=2.0,
        mosaic=0.3, mixup=0.0, copy_paste=0.0, erasing=0.1, fliplr=0.5, scale=0.4,
        close_mosaic=3,
        save_period=2, patience=5, amp=True, verbose=True, exist_ok=True,
    )
print(f'1280px boost done in {(time.time()-t0)/3600:.1f} h')


In [ ]:
# ── Full-dataset polish (20 epochs, close_mosaic=8) ──────────────────────────
# Restores generalisation after 1280px boost; stabilises weights for deploy.
# close_mosaic=8: mosaic off for final 8 epochs — clean images, mimics inference.
# lr0=2e-6: very low — weight stabilisation, not a new learning phase.
# dfl=2.0: carry forward — critical for small-vehicle precision.
# Council A: 20 epochs (up from 15) — more clean-image fine-tuning.
# ~20 min/epoch on T4×2 → ~6.7 hr total.
from ultralytics import YOLO
import shutil, time

start = S3_1280_BEST if S3_1280_BEST.exists() else UPLOADED_S2
assert start.exists(), f'No 1280px checkpoint at {start} — run boost cell first'
assert yaml_path.exists(), 'Run dataset cells first'

t0 = time.time()
if S3P_LAST.exists():
    print(f'Polish: resuming from {S3P_LAST}')
    model = YOLO(str(S3P_LAST)); model.train(resume=True)
else:
    print(f'Polish: starting from {start}')
    model = YOLO(str(start))
    model.train(
        data=str(yaml_path), imgsz=IMGSZ, epochs=20, batch=BATCH, device=DEVICE,
        project=str(RUNS_DIR/'s3p'), name='argus_s3p',
        freeze=0, optimizer='AdamW', lr0=2e-6, lrf=0.1, warmup_epochs=0,
        weight_decay=5e-4, box=9.0, cls=0.3, dfl=2.0,
        mosaic=0.9, close_mosaic=8,
        mixup=0.0, copy_paste=0.05, erasing=0.1, fliplr=0.5, scale=0.4,
        save_period=5, patience=10, amp=True, verbose=True, exist_ok=True,
    )
print(f'Polish done in {(time.time()-t0)/3600:.1f} h')
final = S3P_BEST if S3P_BEST.exists() else S3_1280_BEST
if final.exists():
    shutil.copy2(final, WORK/'argus_final.pt')
    print(f'\n✓ Final model → {WORK}/argus_final.pt')


In [ ]:
# ── Evaluation — mAP50-95 primary (conf=0.001, iou=0.6) ──────────────────────
# conf=0.001: exposes all detections (standard COCO eval practice).
# iou=0.6:    stricter box match — penalises imprecise localisation for TTC.
# PRIMARY: mAP50-95. Target ≥ 0.62 (publication threshold).
from ultralytics import YOLO

candidates = [WORK/'argus_final.pt', S3P_BEST, S3_1280_BEST,
              WORK/'argus_s2.pt', UPLOADED_S2]
EVAL_MODEL = next((c for c in candidates if c.exists()), None)
assert EVAL_MODEL, 'No trained model found'
print(f'Evaluating: {EVAL_MODEL}')

model = YOLO(str(EVAL_MODEL))
metrics = model.val(
    data=str(yaml_path), imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    conf=0.001, iou=0.6, verbose=True,
)

print('\n' + '='*60)
print('  ARGUS Final Validation Results')
print('='*60)
print(f'  mAP50    : {metrics.box.map50:.4f}')
print(f'  mAP50-95 : {metrics.box.map:.4f}  ← PRIMARY (localisation for TTC)')
print()
for name, ap50, ap in zip(CLASSES, metrics.box.ap50, metrics.box.maps):
    flag = '  ← near-miss actor' if name == 'motorcycle' else ''
    print(f'  {name:12s}: AP50={ap50:.3f}  AP50-95={ap:.3f}{flag}')
print('='*60)

moto50 = metrics.box.ap50[1]; moto = metrics.box.maps[1]
if moto50 < 0.70:   print('\n⚠  Motorcycle AP50 < 0.70 — increase IDD/BDD oversample')
elif moto50 < 0.80: print(f'\n→  Motorcycle AP50 {moto50:.3f} — acceptable for publication')
else:               print(f'\n✓  Motorcycle AP50 {moto50:.3f}  AP50-95 {moto:.3f} — excellent')

# TTA sweep
print('\nRunning TTA evaluation...')
mt = model.val(data=str(yaml_path), imgsz=IMGSZ, batch=max(1,BATCH//2),
               device=DEVICE, augment=True, conf=0.001, iou=0.6, verbose=False)
print(f'  TTA mAP50: {mt.box.map50:.4f}  mAP50-95: {mt.box.map:.4f}'
      f'  (gain: +{mt.box.map-metrics.box.map:.4f})')


In [ ]:
# ── Held-out evaluation (locked 2k UA-DETRAC) ─────────────────────────────────
# 2k images locked in Session 1 — NEVER trained on. Domain-shift delta
# between val (dashcam) and held-out (overhead CCTV) is the generalisation
# metric reported in the paper. Target: delta < 5 pp.
from ultralytics import YOLO

if not held_yaml.exists():
    print('Held-out data.yaml not found — re-run UA-DETRAC cell to reconstruct')
else:
    candidates = [WORK/'argus_final.pt', S3P_BEST, S3_1280_BEST, UPLOADED_S2]
    EVAL_MODEL = next((c for c in candidates if c.exists()), None)
    assert EVAL_MODEL, 'No trained model found'

    model = YOLO(str(EVAL_MODEL))
    hm = model.val(data=str(held_yaml), imgsz=IMGSZ, batch=BATCH,
                   device=DEVICE, conf=0.001, iou=0.6, verbose=True)
    vm = model.val(data=str(yaml_path), imgsz=IMGSZ, batch=BATCH,
                   device=DEVICE, conf=0.001, iou=0.6, verbose=False)

    print('\n' + '='*60)
    print('  ARGUS Out-of-Distribution (Held-Out) Evaluation')
    print('='*60)
    print(f'  Validation mAP50    : {vm.box.map50:.4f}')
    print(f'  Held-out  mAP50     : {hm.box.map50:.4f}')
    print(f'  Domain-shift delta  : {hm.box.map50-vm.box.map50:+.4f}')
    print()
    print(f'  Validation mAP50-95 : {vm.box.map:.4f}')
    print(f'  Held-out  mAP50-95  : {hm.box.map:.4f}')
    print('='*60)
    for name, ap50 in zip(CLASSES, hm.box.ap50):
        print(f'  {name:12s}: {ap50:.4f}')
    delta = hm.box.map50 - vm.box.map50
    if abs(delta) < 0.05:
        print('\n✓  Domain shift < 5 pp — strong cross-domain generalisation')
    else:
        print(f'\n⚠  Domain shift {delta:+.4f} — UA-DETRAC augmentation recommended')


In [ ]:
# ── ONNX export + download instructions ──────────────────────────────────────
import shutil
from ultralytics import YOLO

candidates = [WORK/'argus_final.pt', S3P_BEST, S3_1280_BEST, UPLOADED_S2]
FINAL = next((c for c in candidates if c.exists()), None)
assert FINAL, 'No trained model found'

model = YOLO(str(FINAL))
shutil.copy2(FINAL, WORK/'argus_yolo12x_final.pt')
print(f'PyTorch → {WORK}/argus_yolo12x_final.pt')

onnx = model.export(format='onnx', imgsz=640, simplify=True, opset=17, dynamic=False)
shutil.copy2(onnx, WORK/'argus_yolo12x_final.onnx')
print(f'ONNX    → {WORK}/argus_yolo12x_final.onnx')

print('\n' + '='*60)
print('  Download from /kaggle/working/:')
print('  ├── argus_yolo12x_final.pt    drop into ARGUS repo root')
print('  └── argus_yolo12x_final.onnx  CPU/Jetson TRT pipeline')
print()
print('  detection.py:  VehicleDetector("argus_yolo12x_final.pt")')
print('='*60)